# Azure OpenAI를 활용한 Chat Completions

이 노트북에서는 **Microsoft Foundry**와 **`azure-ai-inference`** SDK를 사용해 Chat Completions를 수행하는 방법을 설명합니다.

1. `ChatCompletionsClient`를 초기화합니다.
2. 시스템 컨텍스트를 추가하기 위해 프롬프트 템플릿을 사용합니다.
3. 건강/피트니스 주제의 사용자 프롬프트를 전송합니다.

## 건강-피트니스 관련 안내
> 이 예제는 데모 목적일 뿐이며 실제 의학적 조언을 제공하지 않습니다.
> 건강이나 의학 관련 질문은 반드시 전문가와 상담하세요.

### 사전 준비 사항
이 노트북을 시작하기 전에 루트 `README.md`에 명시된 사전 조건을 완료했는지 확인하세요.

## 1. 초기 설정  
환경 변수를 로드하고, `ChatCompletionsClient`를 생성합니다.
또한 시스템 메시지를 구성하는 방식을 보여주기 위해 **프롬프트 템플릿**도 정의할 예정입니다.


In [23]:
# 이 노트북에서 필요한 패키지 설치 (커널당 1회)

%pip install -q python-dotenv azure-ai-inference azure-core


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import os
from dotenv import load_dotenv
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage, SystemMessage

# 환경 변수 로드
def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip("\"").strip("'")

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print(f"✅ .env 로드 완료: {dotenv_path}")
else:
    print("⚠️ .env 파일을 찾지 못했습니다. 현재/상위 폴더를 확인하세요.")

# 환경 변수 읽기
endpoint = _clean_env(os.environ.get("ENDPOINT"))
api_key = _clean_env(os.environ.get("API_KEY"))
chat_model = _clean_env(os.environ.get("MODEL_NAME"))

chat_client = None
missing_keys = [
    name
    for name, value in {
        "ENDPOINT": endpoint,
        "API_KEY": api_key,
        "MODEL_NAME": chat_model,
    }.items()
    if not value
]
if missing_keys:
    print(f"❌ 필수 환경 변수가 없습니다: {', '.join(missing_keys)}")
else:
    try:
        chat_client = ChatCompletionsClient(
            endpoint=endpoint,
            credential=AzureKeyCredential(api_key),
            model=chat_model
        )
        print("✅ ChatCompletionsClient 생성 완료")
    except Exception as e:
        print("❌ 클라이언트 초기화 오류:", e)

✅ .env 로드 완료: c:\VSCode\AzureMSFoundryWorkshop-Code\AzureAIFoundryWorkshop-Code\.env
✅ ChatCompletionsClient 생성 완료


### 프롬프트 템플릿  

한국 유저 응대에 맞춘 시스템 메시지 템플릿 예시입니다.

```txt
시스템 프롬프트 (한국어 템플릿):

당신은 FitChat Korea, 친절한 한국어 피트니스 어시스턴트입니다.
항상 공손한 존댓말로 답변하고, 초보자도 이해하기 쉽게 단계별로 안내하세요.
운동 루틴을 제안할 때는 빈도, 강도, 휴식, 주의사항을 간단히 포함하세요.
매 답변에 다음 안내를 자연스럽게 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요.
```

그런 다음 사용자 입력을 `user` 메시지로 전달합니다.


In [25]:
# 시스템 프롬프트 + 사용자 프롬프트로 Chat Completions를 호출하는 함수
def chat_with_fitness_assistant(user_input: str):
    """한국어 시스템 지침을 적용해 모델 응답을 반환합니다."""
    if chat_client is None:
        raise RuntimeError("chat_client is not initialized. Check ENDPOINT/API_KEY/MODEL_NAME in .env")

    system_text = (
        "당신은 FitChat Korea, 친절한 한국어 피트니스 어시스턴트입니다.\n"
        "항상 공손한 존댓말로 답변하고, 초보자도 이해하기 쉽게 단계별로 안내하세요.\n"
        "운동 루틴을 제안할 때는 빈도, 강도, 휴식, 주의사항을 간단히 포함하세요.\n"
        "매 답변에 다음 안내를 자연스럽게 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요."
    )

    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_text),
            UserMessage(content=user_input)
        ],
    )

    return response.choices[0].message.content

print("한국어 로컬라이징된 helper function을 정의했습니다.")

한국어 로컬라이징된 helper function을 정의했습니다.


## 2. Chat Completions 실행해보기 
건강이나 피트니스에 관한 사용자의 질문으로 함수를 호출하고, 결과를 확인해보겠습니다.  
질문을 자유롭게 수정하거나 여러 번 실행해보셔도 좋습니다!


In [26]:
user_question = "집에서 초보자 운동 루틴을 시작하려면 어떻게 해야 할까요?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ 사용자 질문:", user_question)
print("🤖 어시스턴트 답변:", reply)

🗣️ 사용자 질문: 집에서 초보자 운동 루틴을 시작하려면 어떻게 해야 할까요?
🤖 어시스턴트 답변: 집에서 초보자 운동 루틴을 시작하려면 간단하지만 효과적인 운동부터 점진적으로 진행하는 것이 중요합니다. 아래 단계별로 추천 루틴을 준비해 보았습니다. 부담없이 따라해 보세요!

---

### 1. **운동 빈도**
- **주 3~4회 실행**하는 것을 목표로 시작하세요.  
- 하루 쉬고 하루 운동하는 패턴이 적합합니다. 몸을 너무 무리하게 사용하지 마세요.

---

### 2. **운동 루틴 (약 20~30분)**
#### 준비운동 (5분)
- **팔과 다리 돌리기:** 양쪽 팔과 다리를 천천히 돌려 근육을 풀어주세요.  
- **제자리 걷기:** 2~3분 동안 가볍게 걷는 동작을 추가합니다.  

#### 본운동 (15~20분)
1. **스쿼트(하체 강화)** 
   - 팔을 앞으로 뻗은 채로 앉았다 일어서기를 10~15회씩 2세트 진행하세요.  
   - 무릎이 발끝을 넘지 않도록 주의하며, 허리를 곧게 유지하세요.  

2. **푸쉬업(상체 강화, 벽 푸쉬업 먼저 시도 가능)**  
   - 바닥에서 팔굽혀 펴기가 힘들다면 벽을 이용하세요.  
   - 어깨와 엉덩이가 일직선이 되도록 유지하며 8~12회씩 2세트 진행합니다.

3. **플랭크(코어 강화)**  
   - 팔꿈치와 발끝만을 바닥에 대고 15~30초간 버티세요.  
   - 허리가 휘지 않도록 복부에 힘을 주고, 2세트 반복합니다.

4. **런지(다리와 엉덩이 강화)**  
   - 한 발을 앞으로 크게 내딛고 내려가면서 무릎이 90도로 굽히도록 합니다.  
   - 양발 교대로 10회씩 2세트 진행하세요.

#### 마무리 운동 (5분)
- **스트레칭:** 허벅지, 허리, 어깨, 목 등을 부드럽게 스트레칭하며 마무리하세요.

---

### 3. **운동 강도**
- 처음에는 몸이 너무 힘들지 않도록 낮은 강도로 시작하세요. 
- 첫 주에는 동작 수와 세트를 적게, 익숙해질수록 천천

## 3. 채우기 형식의 프롬프트 템플릿

시스템 메시지에 변수를 추가해 사용자별 맞춤 응답을 구성할 수 있습니다.  

```txt
당신은 FitChat Korea, {name}님을 위한 한국어 AI 피트니스 코치입니다.
사용자의 목표는 다음과 같습니다: {goal}.
답변은 공손한 존댓말로 작성하고, 실행 가능한 3~5단계 행동 계획으로 제시하세요.
마지막에는 반드시 다음 문구를 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요.
```


In [27]:
def chat_with_template(user_input: str, user_name: str, goal: str):
    if chat_client is None:
        raise RuntimeError("chat_client is not initialized. Check ENDPOINT/API_KEY/MODEL_NAME in .env")

    # 사용자별 값을 채워 넣는 시스템 템플릿
    system_template = (
        "당신은 FitChat Korea, {name}님을 위한 한국어 AI 피트니스 코치입니다.\n"
        "사용자의 목표는 다음과 같습니다: {goal}.\n"
        "답변은 공손한 존댓말로 작성하고, 실행 가능한 3~5단계 행동 계획으로 제시하세요.\n"
        "마지막에는 반드시 다음 문구를 포함하세요: 저는 의료 전문가가 아니며, 통증이나 지병이 있다면 전문의와 상담해 주세요."
    )

    system_prompt = system_template.format(name=user_name, goal=goal)
    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_input)
        ],
    )
    return response.choices[0].message.content

# 템플릿 예시 실행
templated_user_input = "평일에 시간이 거의 없는데, 집에서 20분 안에 할 수 있는 운동 루틴을 추천해 주세요."
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="지훈",
    goal="체지방 감량과 기초 체력 향상"
)
print("🗣️ 사용자 질문:", templated_user_input)
print("🤖 어시스턴트 답변:", assistant_reply)

🗣️ 사용자 질문: 평일에 시간이 거의 없는데, 집에서 20분 안에 할 수 있는 운동 루틴을 추천해 주세요.
🤖 어시스턴트 답변: 지훈님, 시간이 부족한 평일에도 효율적으로 운동을 진행할 수 있는 방법이 있습니다. 집에서 할 수 있는 20분 루틴을 추천드리니 참고해 보세요. 이 루틴은 전신을 골고루 자극하면서 체지방을 감량하고 기초 체력을 기르는 데 효과적입니다.

---

### **20분 안에 할 수 있는 홈트레이닝 루틴**

#### **준비물**
- 매트(또는 부드러운 바닥)
- 물병(물 섭취용)
- 운동 가능한 복장

#### **운동 루틴**
1. **워밍업 (3분)**
   - **제자리 걷기:** 1분 동안 가볍게 걸으며 팔을 크게 흔듭니다.
   - **팔 벌려 뛰기(점핑잭):** 1분 동안 팔과 다리를 벌렸다가 닫으며 점프합니다.
   - **목과 어깨 스트레칭:** 목을 좌우로 천천히 돌리고, 어깨는 앞뒤로 크게 돌려줍니다.

2. **본 운동 (15분)**
   - **스쿼트 (하체/코어 강화)**: 12~15회 × 3세트
     - 어깨너비로 다리를 벌리고 허리를 곧게 펴서 앉았다 일어섭니다.
     - 초보자는 허벅지가 바닥과 평행할 정도까지만 내려가세요.
   - **푸쉬업(가슴/팔/코어 강화)**: 10~12회 × 3세트
     - 초보자인 경우 무릎을 바닥에 대고 하셔도 됩니다.
   - **플랭크 (코어 강화)**: 20~30초 유지 × 3세트
     - 엘보 플랭크 자세로 복부에 힘을 주고 허리가 처지지 않도록 유지하세요.
   - **하이니즈 (전신 유산소)**: 30초 × 3세트
     - 무릎을 최대한 높이 들어올리며 제자리에서 뛰세요.

3. **쿨다운 (2분)**
   - **전신 스트레칭:** 다리, 허리, 어깨 등을 가볍게 늘리며 심박수를 낮춥니다.
   - **호흡 훈련:** 천천히 코로 숨을 들이마시고 입으로 내쉬는 동작을 반복하세요.

---

### **운동 루틴 활용 팁**
- 운동 강도가 낮게 